# Video tools: download, PDF, PowerPoint, slideshow

1. Run **cell 1** once (takes about a minute, it installs the tools).
2. Paste your video or playlist links into **cell 2**, then run it. When it finishes, a zip downloads (or it saves to Drive).
3. Run **cell 3** to flip through the extracted slides here in the notebook.

For each video you get `slides.pdf`, `slides.pptx`, `frames.zip` and `video.mp4`.


In [ ]:
import json
import re
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import numpy as np

# make sure the tools are installed and up to date (safe to re-run)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "yt-dlp[default]", "img2pdf", "python-pptx"],
    check=True,
)

ROOT = Path("/content/archive")
WORK = Path("/content/work")


def safe_name(s, n=40):
    s = re.sub(r'[\\/:*?"<>|\r\n\t]+', " ", s or "").strip()
    return s[:n].strip() or "video"


def ytdlp(*args, **kw):
    cmd = [sys.executable, "-m", "yt_dlp"]
    if shutil.which("node"):
        cmd += ["--js-runtimes", "node"]
    return subprocess.run(cmd + list(args), check=True, **kw)


def list_entries(url):
    """Return [(title or None, url)] for a video link or a whole playlist."""
    if "playlist?list=" not in url:
        return [(None, url)]
    out = ytdlp("--flat-playlist", "-J", url,
                stdout=subprocess.PIPE, text=True, encoding="utf-8").stdout
    info = json.loads(out)
    items = []
    for e in info.get("entries") or []:
        if not e:
            continue
        link = e.get("url") or ""
        if not link.startswith("http"):
            link = "https://www.youtube.com/watch?v=" + e["id"]
        items.append((e.get("title"), link))
    return items


def find_changes(video, threshold, max_frames):
    """Look at 1 frame/second and return the seconds where the screen
    changed and then settled."""
    w, h = 160, 90
    size = w * h
    cmd = ["ffmpeg", "-v", "error", "-i", str(video),
           "-vf", f"fps=1,scale={w}:{h},format=gray",
           "-f", "rawvideo", "-"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE)
    kept, last_kept, prev, pending, t = [], None, None, 0, 0
    while True:
        buf = proc.stdout.read(size)
        if len(buf) < size:
            break
        cur = np.frombuffer(buf, dtype=np.uint8).astype(np.int16)
        if last_kept is None:
            kept.append(t)
            last_kept = cur
        else:
            changed = np.abs(cur - last_kept).mean() > threshold
            settled = np.abs(cur - prev).mean() < 1.0
            pending = pending + 1 if changed else 0
            if changed and (settled or pending >= 8):
                kept.append(t)
                last_kept = cur
                pending = 0
        prev = cur
        t += 1
    proc.wait()
    if len(kept) > max_frames:
        step = len(kept) / max_frames
        kept = [kept[int(i * step)] for i in range(max_frames)]
    return kept


def extract(video, times, frames_dir):
    frames_dir.mkdir(parents=True, exist_ok=True)
    files = []
    for i, t in enumerate(times, 1):
        f = frames_dir / f"{i:04d}.jpg"
        subprocess.run(
            ["ffmpeg", "-v", "error", "-y", "-ss", str(t), "-i", str(video),
             "-frames:v", "1", "-vf", "scale='min(1600,iw)':-2",
             "-q:v", "3", str(f)],
            check=True,
        )
        if f.exists():
            files.append(f)
    return files


def build_pdf(files, out):
    import img2pdf
    out.write_bytes(img2pdf.convert([str(f) for f in files]))


def build_pptx(files, times, out):
    from PIL import Image
    from pptx import Presentation
    from pptx.util import Inches

    w, h = Image.open(files[0]).size
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = int(prs.slide_width * h / w)
    blank = prs.slide_layouts[6]
    for f, t in zip(files, times):
        slide = prs.slides.add_slide(blank)
        slide.shapes.add_picture(str(f), 0, 0,
                                 width=prs.slide_width, height=prs.slide_height)
        slide.notes_slide.notes_text_frame.text = time.strftime(
            "%H:%M:%S", time.gmtime(t))
    prs.save(str(out))


def build_zip(files, times, out):
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for f in files:
            z.write(f, f.name)
        z.writestr("frames.json", json.dumps(
            [{"file": f.name, "t": t} for f, t in zip(files, times)]))


def process(idx, title, url, mode, sensitivity, max_frames):
    shutil.rmtree(WORK, ignore_errors=True)
    WORK.mkdir(parents=True)
    fmt = ("bv*[vcodec^=avc1][height<=1080]+ba[acodec^=mp4a]"
           "/bv*[height<=1080]+ba/b[height<=1080]/b")
    ytdlp("--no-playlist", "-f", fmt, "--merge-output-format", "mp4",
          "-o", str(WORK / "video.%(ext)s"),
          "--print-to-file", "%(title)s", str(WORK / "title.txt"), url)
    video = WORK / "video.mp4"
    if not title:
        title = (WORK / "title.txt").read_text(encoding="utf-8").strip()
    out = ROOT / f"{idx:03d} - {safe_name(title)}"
    out.mkdir(parents=True, exist_ok=True)
    (out / "title.txt").write_text(f"{title}\n{url}\n", encoding="utf-8")

    if mode in ("slides", "both"):
        times = find_changes(video, sensitivity, max_frames)
        files = extract(video, times, WORK / "frames")
        times = times[: len(files)]
        if not files:
            raise RuntimeError("no frames could be extracted")
        build_pdf(files, out / "slides.pdf")
        build_pptx(files, times, out / "slides.pptx")
        build_zip(files, times, out / "frames.zip")
        print(f"   {len(files)} slides found")
    if mode in ("video", "both"):
        shutil.copy(video, out / "video.mp4")


def run_all(urls, mode="both", sensitivity=3.0, max_frames=300,
            save_to_drive=False):
    urls = [u.strip() for u in urls if u.strip() and "PASTE_" not in u]
    if not urls:
        print("Paste at least one link into URLS first.")
        return
    shutil.rmtree(ROOT, ignore_errors=True)
    ROOT.mkdir(parents=True)
    items = []
    for u in urls:
        items += list_entries(u)
    failed = []
    for i, (title, url) in enumerate(items, 1):
        print(f"[{i}/{len(items)}] {title or url}")
        try:
            process(i, title, url, mode, sensitivity, max_frames)
        except Exception as e:
            failed.append((title or url, str(e)[:200]))
            print("   FAILED:", str(e)[:200])
    done = len(items) - len(failed)
    print(f"\nFinished: {done} of {len(items)} worked.")
    for name, err in failed:
        print(" - failed:", name)
    if not done:
        print("If the errors say 'Sign in to confirm you're not a bot', "
              "YouTube is blocking this Colab session. Try again later or "
              "use Runtime > Disconnect and delete runtime, then re-run.")
        return

    if save_to_drive:
        from google.colab import drive
        drive.mount("/content/drive")
        dest = Path("/content/drive/MyDrive/video_archive")
        shutil.copytree(ROOT, dest, dirs_exist_ok=True)
        print("Saved to My Drive > video_archive")
    else:
        zip_path = Path("/content/archive.zip")
        zip_path.unlink(missing_ok=True)
        subprocess.run(["zip", "-r", "-0", "-q", str(zip_path), ROOT.name],
                       cwd=str(ROOT.parent), check=True)
        from google.colab import files
        files.download(str(zip_path))


def browse():
    """Click through the extracted slides of any processed video."""
    import ipywidgets as w
    from IPython.display import Image, display

    dirs = sorted(d for d in ROOT.iterdir() if (d / "frames.zip").exists())
    if not dirs:
        print("No slides yet. Run the previous cell with MODE = 'slides' or 'both'.")
        return
    dd = w.Dropdown(options=[(d.name, str(d)) for d in dirs],
                    description="Video", layout=w.Layout(width="700px"))
    slider = w.IntSlider(min=1, max=1, step=1, description="Slide",
                         layout=w.Layout(width="600px"))
    play = w.Play(interval=1500, min=1, max=1, step=1)
    w.jslink((play, "value"), (slider, "value"))
    out = w.Output()
    state = {}

    def show(_=None):
        if "meta" not in state:
            return
        m = state["meta"][slider.value - 1]
        with out:
            out.clear_output(wait=True)
            stamp = time.strftime("%H:%M:%S", time.gmtime(m["t"]))
            print(f"{slider.value} / {len(state['meta'])}   at {stamp}")
            display(Image(data=state["zip"].read(m["file"]), width=900))

    def load(_=None):
        z = zipfile.ZipFile(Path(dd.value) / "frames.zip")
        state["zip"] = z
        state["meta"] = json.loads(z.read("frames.json"))
        n = len(state["meta"])
        slider.max = n
        play.max = n
        slider.value = 1
        show()

    dd.observe(load, "value")
    slider.observe(show, "value")
    display(dd, w.HBox([play, slider]), out)
    load()


In [ ]:
URLS = [
    "PASTE_A_VIDEO_OR_PLAYLIST_LINK_HERE",
    # add more links on new lines, e.g. "https://www.youtube.com/playlist?list=..."
]

MODE = "both"          # "both", "slides" (PDF + PowerPoint + frames only), or "video" (video only)
SENSITIVITY = 3.0      # screen-change threshold: lower = more slides, higher = fewer
MAX_FRAMES = 300       # most slides kept per video
SAVE_TO_DRIVE = False  # True = save to My Drive > video_archive instead of downloading a zip

run_all(URLS, MODE, SENSITIVITY, MAX_FRAMES, SAVE_TO_DRIVE)


In [ ]:
browse()
